In [ ]:
#| export
# Lazy type-hint evaluation, so hints like `str | None` in route
# signatures work on every Python version.
from __future__ import annotations

In [2]:
#| default_exp web_app


# Step 9: Gallery endpoint + canonical `create_app`

![Step 9 diagram](https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/docs/diagrams/09-step.png)

**Goal:** add a `/gallery` endpoint that returns recent sightings as JSON, a `/sighting/{id}/image` endpoint that serves each cropped bird photo straight from the database — and then a `/gallery.html` page with **buttons**, so humans get a face too.

We define the full factory `create_app` here — `/`, `/health`, `/gallery`, the image route, and the HTML page — plus a `main()` entrypoint so the whole site starts with one command. `bird_watcher/web_app.py` is now complete and deployable.

One wrinkle worth teaching: our sightings carry raw JPEG `bytes`, and **bytes can't travel inside JSON**. So the gallery sends a *link* for each photo, and the browser fetches the image from its own endpoint — exactly how every real website does it.

## Step 9.0 — Setup

No secrets this lesson — just the path to the database that lesson 6 filled with sightings.

In [3]:
from pathlib import Path

DATA_FOLDER = Path("../data")
DB_PATH = DATA_FOLDER / "birds.db"

print(f"Database file: {DB_PATH}")


Database file: ../data/birds.db


In [4]:
from bird_watcher.persistence import SqliteSightingStorage

storage = SqliteSightingStorage(
    db_path=DB_PATH, 
    cooldown_minutes=3)

## Step 9.1 — The factory + all four endpoints

In [ ]:
#| export
from fastapi import FastAPI, HTTPException, Response
from fastapi.responses import HTMLResponse

from bird_watcher.persistence import Sighting, SightingStorage


def create_app(storage: SightingStorage) -> FastAPI:
    """Build the full Bird Watcher FastAPI app.

    Endpoints:
        GET /                       — landing page metadata
        GET /health                 — uptime check
        GET /gallery                — recent sightings as JSON (the machine face)
        GET /gallery.html           — the same sightings as a web page (the human face)
        GET /sighting/{id}          — one sighting's metadata
        GET /sighting/{id}/image    — one cropped bird photo, straight from storage

    Args:
        storage: any SightingStorage — the app only knows the contract,
            never which database sits behind it.

    Returns:
        A configured FastAPI app, ready to be served with uvicorn.
    """
    app = FastAPI(title="Bird Watcher", version="0.0.0")

    def to_json(sighting: Sighting) -> dict:
        """One sighting as a JSON-safe dict.

        Bytes can't travel inside JSON, so the photo travels as a link."""
        return {
            "id": sighting.id,
            "timestamp": sighting.timestamp,
            "species": sighting.species,
            "confidence": sighting.confidence,
            "submitted_by": sighting.submitted_by,
            "image_url": f"/sighting/{sighting.id}/image",
        }

    @app.get("/")
    def index() -> dict:
        """Landing page."""
        return {"app": "bird-watcher", "status": "watching"}

    @app.get("/health")
    def health() -> dict:
        """Health check — useful for uptime monitoring."""
        return {"status": "ok"}

    @app.get("/gallery")
    def gallery(species: str | None = None, submitted_by: str | None = None,
                limit: int = 50, offset: int = 0) -> dict:
        """Recent sightings, newest first.

        Filter with ?species= and/or ?submitted_by=, page with ?limit= and ?offset=."""
        sightings = storage.get_sightings(
            species=species, submitted_by=submitted_by, limit=limit, offset=offset
        )
        return {"count": len(sightings), "sightings": [to_json(s) for s in sightings]}

    @app.get("/gallery.html")
    def gallery_page() -> HTMLResponse:
        """The human face: HTML + JavaScript that calls our own JSON API (see Step 9.5).

        We read the file on EVERY request — so edit tutorials/static/gallery.html,
        refresh the browser, and see the change. No restart."""
        return HTMLResponse(find_gallery_html().read_text())

    @app.get("/sighting/{sighting_id}")
    def get_sighting(sighting_id: int) -> dict:
        """One sighting's metadata, without the image bytes."""
        sighting = storage.get_sighting(sighting_id)
        if sighting is None:
            raise HTTPException(status_code=404, detail=f"no sighting #{sighting_id}")
        return to_json(sighting)

    @app.get("/sighting/{sighting_id}/image")
    def get_sighting_image(sighting_id: int) -> Response:
        """Serve one sighting's photo straight from storage."""
        sighting = storage.get_sighting(sighting_id)
        if sighting is None:
            raise HTTPException(status_code=404, detail=f"no sighting #{sighting_id}")

        return Response(content=sighting.image_bytes, media_type="image/jpeg")

    return app


## Step 9.2 — Smoke-test the app

In [ ]:
from fastapi.testclient import TestClient
from bird_watcher.web_app import create_app

app = create_app(storage)  # the SqliteSightingStorage from Step 9.0
client = TestClient(app)

r = client.get("/")
assert r.status_code == 200
assert r.json()["app"] == "bird-watcher"

r = client.get("/health")
assert r.status_code == 200

r = client.get("/gallery")
assert r.status_code == 200
data = r.json()
assert "count" in data and "sightings" in data

if data["sightings"]:
    first = data["sightings"][0]

    r = client.get(f"/sighting/{first['id']}")
    assert r.status_code == 200
    assert r.json()["species"] == first["species"]

    img = client.get(first["image_url"])
    assert img.status_code == 200
    assert img.headers["content-type"] == "image/jpeg"
    print(f"thumbnail for sighting #{first['id']}: {len(img.content)/1000:.1f} KB of JPEG")

print(f"✅ all endpoints respond. /gallery returned {data['count']} sightings.")

## Step 9.3 — Explore it in your browser

The TestClient proved the routes *work* — now let's *see* them. Same trick as lesson 8: run the real server on a background thread, then click through the URLs.

Each URL below demonstrates one capability:

- `/gallery` — the sightings as JSON
- `/gallery?limit=2` — the same route, steered by a **query parameter** (change the number, refresh, watch the response change)
- `/sighting/1/image` — not JSON at all: the route answers with a JPEG, so the browser shows you the bird
- `/gallery.html` — the human face: a real web page with buttons (built in Step 9.5 below)
- `/docs` — the explorer sandbox FastAPI generated from our code: expand a route, hit **Try it out**, and fire requests with buttons instead of typing URLs

That `/docs` page is worth a beat: we never wrote it. FastAPI read our route functions — names, parameters, docstrings — and built an interactive test console from them. Good names and type hints aren't just style; machines read them too.

In [ ]:
import threading
import time

import uvicorn

server = uvicorn.Server(uvicorn.Config(app, port=8123, log_level="info"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(1)

print("Click through the capabilities:")
print("  http://localhost:8123/gallery")
print("  http://localhost:8123/gallery?limit=2")
print("  http://localhost:8123/sighting/1/image")
print("  http://localhost:8123/gallery.html")
print("  http://localhost:8123/docs")

In [ ]:
server.should_exit = True
time.sleep(1)
print("Server stopped.")

## Step 9.4 — Swap the storage, keep the website

Look back at `create_app`'s signature: it never asked for SQLite. It asked for *any* `SightingStorage` — the contract we wrote down in lesson 6.

To prove the website really doesn't care, let's hand it a **fake**: a store that keeps sightings in a plain Python list. No database, no files, no SQL. It only has to keep the same three promises — `post_sighting`, `get_sightings`, `get_sighting` — and the cooldown logic from lesson 6 comes along for free, because that was written once in the base class.

The website can't tell the fake from the real database, because both keep the promise. Two things fall out of that:

- **Testing gets easy** — we can test every route without a database file on disk.
- **Swapping gets easy** — moving from SQLite to a big-kid database server later means writing one new subclass and changing one line where the app is built. `create_app` doesn't change at all.

In [ ]:
from dataclasses import dataclass, field

from bird_watcher.persistence import Sighting, SightingStorage


@dataclass
class FakeSightingStorage(SightingStorage):
    """Keeps sightings in a plain list — no database, no files, no SQL."""
    sightings: list[Sighting] = field(default_factory=list)

    def post_sighting(self, sighting: Sighting, debug_prn: bool = False):
        sighting.id = len(self.sightings) + 1
        self.sightings.insert(0, sighting)  # newest first, like the real store

    def get_sightings(self, species: str | None = None, limit: int = 100, offset: int = 0,
                      submitted_by: str | None = None) -> list[Sighting]:
        rows = [
            s for s in self.sightings
            if (species is None or s.species == species)
            and (submitted_by is None or s.submitted_by == submitted_by)
        ]
        return rows[offset:offset + limit]

    def get_sighting(self, sighting_id: int) -> Sighting | None:
        return next((s for s in self.sightings if s.id == sighting_id), None)


fake = FakeSightingStorage(cooldown_minutes=0)
fake.post_sighting(Sighting(species="blue jay", confidence=0.97, image_bytes=b"\xff\xd8fake-jpeg"))

fake_client = TestClient(create_app(fake))

assert fake_client.get("/gallery").json()["count"] == 1
assert fake_client.get("/gallery?species=blue jay").json()["count"] == 1
assert fake_client.get("/gallery?species=owl").json()["count"] == 0
assert fake_client.get("/sighting/1/image").content == b"\xff\xd8fake-jpeg"
assert fake_client.get("/sighting/99").status_code == 404

print("✅ same website, zero SQL — the fake keeps the SightingStorage promise, so create_app can't tell the difference.")

## Step 9.5 — Buttons for humans

So far every route answers in JSON — great for programs, unreadable for grandma. Time for the **human face**: a web page with buttons.

Here's the trick, and it's the same trick as the image links: **the server can hand out the client.** One route (`/gallery.html`) returns a page of HTML. Inside that page lives a little JavaScript, and that JavaScript calls the *same* `/gallery` endpoint you've been testing all lesson. Count the clients we've now pointed at our API: the browser's URL bar, Python's `requests`, `TestClient`, your phone — and now the page's own JavaScript. `fetch(...)` is just `requests.get(...)` in browser clothes.

The page itself lives in its own file — [`tutorials/static/gallery.html`](static/gallery.html) — because HTML deserves an HTML file: your editor highlights it properly, and it isn't squashed inside a Python string. Open it now and find these three things:

- **The buttons are query parameters wearing costumes.** Clicking *blue jay* just re-fetches `/gallery?species=blue jay` — the exact URL you typed by hand in Step 9.3. Buttons aren't magic; they're saved URLs.
- **The species buttons build themselves from the data.** The page asks the API which birds exist, then makes one button per species. New bird in the database → refresh → new button. We never hard-code a bird name.
- **The `image_url` links pay off.** In Step 9.1 we said "bytes can't travel in JSON, so each photo travels as a link." Now you see why: the browser follows every `<img src="/sighting/3/image">` on its own, and the page fills with birds.

The route reads the file **on every request**, which gives you the best dev loop there is: edit the HTML, refresh the browser, see the change — no server restart. The one wrinkle is that our code runs in two worlds (a notebook cell inside `tutorials/`, and the exported `bird_watcher/web_app.py`), and the path to the file looks different from each — so `find_gallery_html` below checks both. That's also our deploy contract: the app ships as the whole repo folder, not just the `bird_watcher/` package.

One honest caveat: the page builds HTML with template literals and `innerHTML`, which is fine for a hobby page showing *our own* data. Real sites escape user-supplied text first — same lesson as the SQL `?` placeholders in lesson 6, just aimed at the browser instead of the database.

In [ ]:
#| export
from pathlib import Path


def find_gallery_html() -> Path:
    """Locate tutorials/static/gallery.html from either world this code runs in.

    The same code runs in two places, and "where am I?" differs:
      - as a notebook cell — no `__file__`, working directory is `tutorials/`
      - as the exported module — `__file__` is `bird_watcher/web_app.py`
    """
    here = Path(__file__).parent if "__file__" in globals() else Path.cwd()

    candidates = [
        here / "static" / "gallery.html",                       # notebook: cwd is tutorials/
        here.parent / "tutorials" / "static" / "gallery.html",  # module: up from bird_watcher/
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"gallery.html not found — looked in: {[str(c) for c in candidates]}")

In [ ]:
print(f"gallery page lives at: {find_gallery_html()}")

# the page is just another route — so it tests like one.
# it even works against the fake storage: the HTML doesn't care either.
r = client.get("/gallery.html")
assert r.status_code == 200
assert "text/html" in r.headers["content-type"]
assert "Bird Watcher" in r.text

r = fake_client.get("/gallery.html")
assert r.status_code == 200

print("✅ /gallery.html serves the page — re-run the server cell in Step 9.3,")
print("   then open  http://localhost:8123/gallery.html  and click your birds.")
print("   then EDIT tutorials/static/gallery.html, refresh the browser, and watch it change.")

## Step 9.6 — One command to run it anywhere

Until now the server only exists while a notebook cell is running. A *deployable* app needs a single command that starts everything — the same command on your laptop, a Raspberry Pi, or a rented server.

That command is a `main()` function that does the wiring we've been doing by hand in Step 9.0: build the storage, hand it to `create_app`, give the app to uvicorn. Configuration comes from **environment variables** (remember the `.env` lesson?) so the same code runs anywhere — only the settings change.

We register `main` as a script in `pyproject.toml`:

```toml
[project.scripts]
bird-watcher-web = "bird_watcher.web_app:main"
```

After that, starting your whole bird website — API, gallery page, photos — is:

```bash
uv run bird-watcher-web
```

That's the deploy story: copy the project to any machine with Python, point `BIRDS_DB` at your database file, run one command.

In [ ]:
#| export
import os

from bird_watcher.persistence import SqliteSightingStorage


def main():
    """Start the Bird Watcher web server — the one command a deploy runs.

    Settings come from environment variables, so the same code runs on a
    laptop and on a server:

        BIRDS_DB          — path to the SQLite file  (default: data/birds.db)
        COOLDOWN_MINUTES  — sighting cooldown        (default: 3)
        PORT              — port to listen on        (default: 8123)
    """
    import uvicorn

    storage = SqliteSightingStorage(
        db_path=os.environ.get("BIRDS_DB", "data/birds.db"),
        cooldown_minutes=int(os.environ.get("COOLDOWN_MINUTES", "3")),
    )

    uvicorn.run(
        create_app(storage),
        host="0.0.0.0",  # answer anyone on the network, not just this machine
        port=int(os.environ.get("PORT", "8123")),
    )

## Acceptance criterion

- `/`, `/health`, `/gallery`, `/gallery.html`, `/sighting/{id}`, and `/sighting/{id}/image` all respond with 200 against the real SQLite storage.
- The same `create_app` passes every check against `FakeSightingStorage` — no database file needed.
- `uv run bird-watcher-web` starts the whole site with one command.

## What's next

**Step 10:** open [10-digest.ipynb](10-digest.ipynb) — we'll post a daily summary to Slack.